In [ ]:
#Link model : https://www.kaggle.com/models/trungnguyen2710t/tron-advanced

In [ ]:
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Số GPU         : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}        : {torch.cuda.get_device_name(i)}")
    mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  VRAM {i}       : {mem:.1f} GB")

In [ ]:
!nvidia-smi

In [ ]:
import os
import gc
import json
import math
import random
import shutil
import itertools
from datetime import datetime
from functools import partial
from pathlib import Path
from random import sample
import random
import numpy as np
import polars as plx
import pytorch_lightning as py_light
import torch
from torch import (
    cat, concat, cumsum, diag, exp, flip, inf,
    log, logical_and, logical_or, nn,
    sigmoid, softmax, stack, tensor, tile, topk, where,
)
from torch.nn import CrossEntropyLoss, Dropout
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
from pytorch_lightning.trainer.trainer import Trainer
from pytorch_lightning.callbacks import (
    Callback, EarlyStopping, ModelCheckpoint,
)
from pytorch_lightning.loggers import CSVLogger

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

try:
    import cudf_polars
    print("cudf_polars loaded — Polars sẽ chạy trên GPU!")
except ImportError:
    print("cudf_polars không có — Polars chạy trên CPU (bình thường).")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Đảm bảo tính toán trên GPU cũng đồng nhất (Deterministic)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)


In [ ]:
def multiply_head_with_embedding(prediction_head, embeddings):
    return prediction_head.matmul(embeddings.transpose(-1, -2))


def lookup_and_multiply(prediction_head, positives, uniform_negatives,
                        in_batch_negatives, embedding_layer, sampling_style):
    positive_logits = multiply_head_with_embedding(
        prediction_head.unsqueeze(-2),
        embedding_layer(positives).unsqueeze(-2)
    ).squeeze(-1)

    if sampling_style == "eventwise":
        uniform_negative_logits = multiply_head_with_embedding(
            prediction_head.unsqueeze(-2),
            embedding_layer(uniform_negatives)
        ).squeeze(-2)
    else:
        uniform_negative_logits = multiply_head_with_embedding(
            prediction_head, embedding_layer(uniform_negatives)
        )

    in_batch_negative_logits = multiply_head_with_embedding(
        prediction_head, embedding_layer(in_batch_negatives)
    )
    negative_logits = concat([uniform_negative_logits, in_batch_negative_logits], dim=-1)
    return positive_logits, negative_logits


ce_loss = CrossEntropyLoss(reduction="none")


def _elementwise_sampled_softmax_loss(pos_logits, neg_logits, mask, target):
    sm_logits = cat((pos_logits, neg_logits), dim=-1)
    shape = sm_logits.shape
    return ce_loss(sm_logits.reshape([-1, shape[-1]]), target).reshape([shape[0], shape[1]]) * mask


def sampled_softmax_loss(pos_logits, neg_logits, mask, device="cpu"):
    target = tensor([0], device=device).tile(mask.numel())
    return _elementwise_sampled_softmax_loss(pos_logits, neg_logits, mask, target).sum() / mask.sum()


def bce_loss(pos_logits, neg_logits, mask, epsilon=1e-10, device="cpu"):
    loss = log(1. + exp(-pos_logits) + epsilon) + log(1. + exp(neg_logits) + epsilon).mean(-1, keepdim=True)
    return (loss * mask.unsqueeze(-1)).sum() / mask.sum()


def _diff_logits(pos_logits, neg_logits):
    return pos_logits - neg_logits


def _elementwise_bpr_max_loss_per_negative(pos_logits, neg_logits):
    logits_diff = sigmoid(_diff_logits(pos_logits, neg_logits))
    s_j = softmax(neg_logits - neg_logits.max(dim=-1)[0].unsqueeze(-1), dim=-1)
    return s_j * logits_diff


def _bpr_max_loss_unregulized(pos_logits, neg_logits, mask):
    bpr_max_loss_per_element = -log(_elementwise_bpr_max_loss_per_negative(pos_logits, neg_logits).sum(dim=-1))
    return bpr_max_loss_per_element, (bpr_max_loss_per_element * mask).sum() / mask.sum()


def _bpr_max_loss_regularization(neg_logits, penalty, mask):
    regularization = penalty * (softmax(neg_logits, dim=-1) * neg_logits * neg_logits).sum(dim=-1)
    return (regularization * mask).sum() / mask.sum()


def bpr_max_loss(penalty, pos_logits, neg_logits, mask, device="cpu"):
    _, unregulized = _bpr_max_loss_unregulized(pos_logits, neg_logits, mask)
    return unregulized + _bpr_max_loss_regularization(neg_logits, penalty, mask)


def calc_loss(loss_fn, x_hat, labels, uniform_negatives, in_batch_negatives,
              mask, embeddings, sampling_style, final_activation,
              topk_sampling=False, topk_sampling_k=1000, device="cpu"):
    pos_logits, neg_logits = lookup_and_multiply(
        x_hat, labels, uniform_negatives, in_batch_negatives, embeddings, sampling_style
    )
    if topk_sampling:
        neg_logits, _ = torch.topk(neg_logits, k=topk_sampling_k, dim=-1)
    pos_scores, neg_scores = final_activation(pos_logits), final_activation(neg_logits)
    return loss_fn(pos_scores, neg_scores, mask, device=device)


In [ ]:
import itertools
from random import sample

import numpy as np

def _uniform_negatives(num_items, shape):
    return np.random.randint(1, num_items + 1, shape)


def _uniform_negatives_session_rejected(num_items, shape, in_session_items):
    negatives = []
    for _ in range(np.prod(shape)):
        negative = np.random.randint(1, num_items + 1)
        while negative in in_session_items:
            negative = np.random.randint(1, num_items + 1)
        negatives.append(negative)
    return np.array(negatives).reshape(shape)


def _infer_shape(session_len, num_uniform_negatives, sampling_style):
    if sampling_style == "eventwise":
        return [session_len, num_uniform_negatives]
    elif sampling_style == "sessionwise":
        return [num_uniform_negatives]
    return []


def sample_uniform(num_items, shape, in_session_items, reject_session_items):
    if reject_session_items:
        return _uniform_negatives_session_rejected(num_items, shape, in_session_items)
    return _uniform_negatives(num_items, shape)


def sample_uniform_negatives_with_shape(clicks, num_items, session_len,
                                        num_uniform_negatives, sampling_style,
                                        reject_session_items):
    in_session_items = set(clicks)
    shape = _infer_shape(session_len, num_uniform_negatives, sampling_style)
    if shape:
        return sample_uniform(num_items, shape, in_session_items, reject_session_items)
    return np.array([])


def sample_in_batch_negatives(batch_positives, num_in_batch_negatives,
                               batch_session_len, reject_session_items):
    in_batch_negatives = []
    positive_indices = [0] + list(itertools.accumulate(batch_session_len))
    if reject_session_items:
        for i in range(len(positive_indices) - 1):
            candidates = (batch_positives[:positive_indices[i]]
                          + batch_positives[positive_indices[i + 1]:])
            in_batch_negatives.append(sample(candidates, num_in_batch_negatives))
    else:
        for _ in batch_session_len:
            in_batch_negatives.append(sample(batch_positives, num_in_batch_negatives))
    return in_batch_negatives


In [ ]:
class BehaviorTemporalBias(nn.Module):
    """
    Behavior-Temporal Bias module.
    Learns a bias B[b_i, b_j, bucket(Δt)] to add to attention scores.
    """
    # Breakpoints tính bằng GIÂY (15 mốc -> 16 buckets)
    BREAKPOINTS_SEC = [1, 10, 60, 300, 1800, 7200, 21600, 43200,
                       86400, 172800, 345600, 604800, 864000, 1209600, 1814400]
    
    def __init__(self, n_behaviors=3, n_buckets=16, n_heads=1):
        super().__init__()
        self.n_b = n_behaviors + 1  # +1 cho padding (index 0)
        self.n_buckets = n_buckets
        self.n_heads = n_heads
        
        # Tensor bias learnable: shape (n_heads, N_b, N_b, N_k)
        self.bias = nn.Parameter(torch.zeros(n_heads, self.n_b, self.n_b, n_buckets))
        nn.init.normal_(self.bias, mean=0.0, std=0.02)
        
        # Đăng ký breakpoints làm buffer (tính bằng milliseconds)
        bp_ms = torch.tensor([b * 1000 for b in self.BREAKPOINTS_SEC], dtype=torch.long)
        self.register_buffer('breakpoints', bp_ms)
    
    def bucket_time(self, delta_t_ms):
        delta_t_ms = delta_t_ms.abs()
        bucket_idx = torch.bucketize(delta_t_ms, self.breakpoints, right=False)
        return bucket_idx.clamp(0, self.n_buckets - 1)
    
    def forward(self, b_seq, timestamps):
        B, T = b_seq.shape
        ts = timestamps.unsqueeze(2)     # (B, T, 1)
        delta_t = (ts - ts.transpose(1, 2)).abs()  # (B, T, T)
        
        t_bucket = self.bucket_time(delta_t)
        
        b_i = b_seq.unsqueeze(2).expand(-1, -1, T)  # query behavior
        b_j = b_seq.unsqueeze(1).expand(-1, T, -1)  # key behavior
        
        # Lấy bias cho từng head
        bias_values = self.bias[:, b_i, b_j, t_bucket]  # (H, B, T, T)
        bias_values = bias_values.permute(1, 0, 2, 3)   # (B, H, T, T)
        
        return bias_values.reshape(B * self.n_heads, T, T)



In [ ]:
class DynamicPositionEmbedding(torch.nn.Module):

    def __init__(self, max_len, dimension):
        super().__init__()
        self.max_len = max_len
        self.embedding = nn.Embedding(max_len, dimension)
        self.register_buffer('pos_indices_const', torch.arange(0, max_len, dtype=torch.int))

    def forward(self, x, device='cpu'):
        seq_len = x.shape[1]
        return self.embedding(self.pos_indices_const[-seq_len:]) + x


def sample_eval_candidates(target, rated_set, num_items, n_neg=100):
    candidates = [int(target)]
    while len(candidates) < n_neg + 1:
        neg = random.randint(1, int(num_items))
        if neg not in rated_set and neg != target:
            candidates.append(neg)
    return candidates

In [ ]:
class SASRec(py_light.LightningModule):

    def __init__(self, hidden_size, dropout_rate, max_len, num_items, batch_size,
                 sampling_style, topk_sampling=False, topk_sampling_k=1000,
                 learning_rate=0.001, num_layers=2, loss='bce', bpr_penalty=None,
                 optimizer='adam', output_bias=False,   share_embeddings=True,
                 final_activation=False, eval_num_negatives=99, eval_cutoff=10,
                 n_behaviors=3, n_bt_buckets=16):
        super().__init__()
        self.learning_rate = learning_rate
        self.hidden_size = hidden_size
        self.dropout_rate = dropout_rate
        self.num_items = num_items
        self.batch_size = batch_size
        self.num_layers = num_layers
        self.max_len = max_len
        self.output_bias = output_bias
        self.share_embeddings = share_embeddings
        self.register_buffer('future_mask_const',
                             torch.triu(torch.ones(max_len, max_len) * float('-inf'), diagonal=1))
        self.register_buffer('seq_diag_const',
                             ~diag(torch.ones(max_len, dtype=torch.bool)))
        self.register_buffer('bias_ones',
                             torch.ones([batch_size, max_len, 1]))

        emb_dim = hidden_size + 1 if (output_bias and share_embeddings) else hidden_size
        self.item_embedding = nn.Embedding(num_items + 1, emb_dim, padding_idx=0)
        self.positional_embedding_layer = DynamicPositionEmbedding(max_len, hidden_size)

        torch.nn.init.xavier_uniform_(self.item_embedding.weight.data)
        torch.nn.init.xavier_uniform_(self.positional_embedding_layer.embedding.weight.data)

        if share_embeddings:
            self.output_embedding = self.item_embedding
        elif output_bias:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size + 1, padding_idx=0)
        else:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)

        self.norm = nn.LayerNorm([hidden_size])
        self.input_dropout = Dropout(dropout_rate)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size, nhead=1, dim_feedforward=hidden_size,
            dropout=dropout_rate, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, norm=self.norm)
        self.merge_attn_mask = True
        self.final_activation = nn.ELU(0.5) if final_activation else nn.Identity()
        self.bt_bias = BehaviorTemporalBias(n_behaviors=n_behaviors, n_buckets=n_bt_buckets, n_heads=1)

        self.loss_fn = loss
        if loss == 'bce':
            self.loss = bce_loss
        elif loss == 'ssm':
            self.loss = sampled_softmax_loss
        elif loss == 'bpr-max':
            if bpr_penalty is None:
                raise ValueError('bpr_penalty must be provided for bpr_max loss')
            self.loss = partial(bpr_max_loss, bpr_penalty)
        else:
            raise ValueError('Loss function not supported')

        self.sampling_style = sampling_style
        self.topk_sampling = topk_sampling
        self.topk_sampling_k = topk_sampling_k
        self.optimizer = optimizer
        self.eval_num_negatives = eval_num_negatives
        self.eval_cutoff = eval_cutoff
        self.save_hyperparameters()

    def merge_attn_masks(self, padding_mask):
        batch_size = padding_mask.shape[0]
        seq_len = padding_mask.shape[1]
        # future_mask_const: 0.0 for past/current, -inf for future (upper triangle)
        future_masks = self.future_mask_const[:seq_len, :seq_len]  # (T, T)
        if not self.merge_attn_mask:
            return future_masks  # (T, T)
        
        # padding_mask: 1.0 = valid token, 0.0 = padding
        # is_pad: True where padding
        is_pad = ~padding_mask.bool()  # (B, T)
        
        # Padding keys (columns): shape (B, 1, T) -> broadcast to (B, T, T)
        # -1e9 for padding key positions, 0.0 for valid
        pad_key_mask = is_pad.unsqueeze(1).float() * -1e9
        
        # Combine: future_masks (T, T) + padding key mask (B, 1, T)
        # Broadcast: (T, T) + (B, 1, T) -> (B, T, T)
        combined = future_masks.unsqueeze(0) + pad_key_mask
        return combined

    def forward(self, item_indices, mask, b_seq=None, timestamps=None):
        att_mask = self.merge_attn_masks(mask)
        if b_seq is not None and timestamps is not None:
            bt_bias = self.bt_bias(b_seq, timestamps)
            att_mask = att_mask + bt_bias

        items = (self.item_embedding(item_indices)[:, :, :-1]
                 if self.output_bias and self.share_embeddings
                 else self.item_embedding(item_indices))
        x = items * np.sqrt(self.hidden_size)
        x = self.positional_embedding_layer(x)
        x = self.encoder(self.input_dropout(x), att_mask)
        return concat([x, self.bias_ones[:x.size(0)]], dim=-1) if self.output_bias else x

    def training_step(self, batch, _):
        x_hat = self.forward(batch["clicks"], batch["mask"], batch.get("b_seq"), batch.get("timestamps"))
        train_loss = calc_loss(
            self.loss, x_hat, batch["labels"], batch["uniform_negatives"],
            batch["in_batch_negatives"], batch["mask"], self.output_embedding,
            self.sampling_style, self.final_activation,
            self.topk_sampling, self.topk_sampling_k, self.device
        )
        self.log("train_loss", train_loss, on_step=True, on_epoch=True, prog_bar=True)
        return train_loss

    def validation_step(self, batch, _batch_idx):
        x_hat = self.forward(batch['clicks'], batch['mask'], batch.get('b_seq'), batch.get('timestamps'))
        last_hidden = x_hat[:, -1, :]
        batch_size = last_hidden.shape[0]
        hr10 = hr20 = ndcg10 = ndcg20 = 0.0

        for i in range(batch_size):
            target = int(batch['target'][i].item())
            rated  = batch['rated'][i]
            candidates = sample_eval_candidates(target, rated, self.num_items, self.eval_num_negatives)
            cand_tensor = torch.tensor(candidates, dtype=torch.long, device=self.device)
            scores = torch.matmul(last_hidden[i], self.output_embedding(cand_tensor).transpose(0, 1))
            rank = int((scores > scores[0]).to(torch.int).sum().item())
            if rank < 10:
                hr10   += 1.0
                ndcg10 += 1.0 / math.log2(rank + 2)
            if rank < 20:
                hr20   += 1.0
                ndcg20 += 1.0 / math.log2(rank + 2)

        denom = max(batch_size, 1)
        self.log('val_hr10',   hr10   / denom, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val_ndcg10', ndcg10 / denom, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val_hr20',   hr20   / denom, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val_ndcg20', ndcg20 / denom, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        if self.optimizer == 'adam':
            return torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        elif self.optimizer == 'adagrad':
            return torch.optim.Adagrad(self.parameters(), lr=self.learning_rate)
        raise ValueError('Optimizer not supported, please use adam or adagrad')


In [ ]:
#Load model tốt nhất
model = SASRec.load_from_checkpoint("/kaggle/working/sasrec_best.ckpt")
num_items = model.hparams.num_items
config = {
    "max_session_length": model.hparams.max_len
}
model.eval()

In [ ]:
def predict_single_session(model, items, types, timestamps, config, num_items, top_k=20, device="cpu"):
    """
    Dự đoán top-K items tiếp theo cho một session duy nhất.
    
    Args:
        model     : SASRec model đã load checkpoint.
        items     : List ID sản phẩm của session (ID gốc bắt đầu từ 0).
        types     : List loại hành vi tương ứng (ví dụ: ['clicks', 'carts', 'clicks'] hoặc [1, 2, 1]).
        timestamps: List mốc thời gian (milliseconds) tương ứng.
        config    : dict cấu hình (cần 'max_session_length').
        num_items : Tổng số sản phẩm trong catalog.
        top_k     : Số sản phẩm gợi ý.
        device    : 'cuda' hoặc 'cpu'.
        
    Returns:
        top_preds : List top-K ID sản phẩm gợi ý (đã trừ 1 về ID gốc).
        top_scores: List điểm số tương ứng.
    """
    model = model.to(device)
    model.eval()
    
    max_len = config["max_session_length"]
    
    # 1. Chuẩn bị items (Shift + 1 để tránh trùng padding index 0)
    items_shifted = [x + 1 for x in items]
    
    # 2. Chuẩn bị types (Map chuỗi sang số nguyên)
    TYPE_MAP = {'clicks': 1, 'carts': 2, 'orders': 3}
    types_mapped = [TYPE_MAP.get(str(t), t) for t in types]
    
    # 3. Cắt chuỗi theo max_session_length
    items_shifted = items_shifted[-max_len:]
    types_mapped = types_mapped[-max_len:]
    timestamps = timestamps[-max_len:]
    
    # 4. Padding về max_len
    seq_len = len(items_shifted)
    pad = max_len - seq_len
    
    padded_items = [0] * pad + items_shifted
    padded_types = [0] * pad + types_mapped
    padded_ts = [0] * pad + timestamps
    
    # 5. Chuyển sang Tensor và thêm batch dimension (B=1)
    seq_tensor = torch.tensor([padded_items], dtype=torch.long, device=device)
    b_seq_tensor = torch.tensor([padded_types], dtype=torch.long, device=device)
    ts_tensor = torch.tensor([padded_ts], dtype=torch.long, device=device)
    mask = (seq_tensor != 0).float()
    
    # 6. Chạy forward pass để tính toán logits
    with torch.no_grad():
        x_hat = model(seq_tensor, mask, b_seq_tensor, ts_tensor)
        last_hidden = x_hat[:, -1, :]
        
        # Lấy embedding của toàn bộ items (chừa index 0)
        all_item_ids = torch.arange(1, num_items + 1, dtype=torch.long, device=device)
        all_emb = model.output_embedding(all_item_ids)
        
        # Nhân ma trận tính scores cho từng item
        scores = torch.matmul(last_hidden, all_emb.T).squeeze(0)
        
        # Loại bỏ các item đã tương tác khỏi danh sách gợi ý
        for item in items_shifted:
            if 0 < item <= num_items:
                scores[item - 1] = -float('inf')
                
        # Lấy top-K gợi ý
        top_scores, top_indices = torch.topk(scores, k=top_k)
        
        top_preds = (all_item_ids[top_indices] - 1).cpu().tolist()
        top_scores = top_scores.cpu().tolist()
        
    return top_preds, top_scores

In [ ]:
items_example = [1023, 4522]
types_example = ['clicks', 'carts']
ts_example = [1685000000000, 1685000005000]
top_preds, top_scores = predict_single_session(model, items_example, types_example, ts_example, config, num_items, top_k=20, device='cpu')
print('Top-20 predicted items:', top_preds)